# DPR processor example with Prefect+Dask

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-520

See the associated:

  * Python module: [dpr_processor_example.py](./dpr_processor_example.py)
  * YAML file: [dpr_processor_example.yaml](./dpr_processor_example.yaml)

**NOTE: This notebook is not run from the ci/cd because we have random errors with eopf.**

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"


Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Get existing dask cluster: '6a4f2571039248ca858ed7b49287cdef'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/6a4f2571039248ca858ed7b49287cdef/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 1/2
Dask workers for 'dask-eopf' are up: 2/2


In [3]:
# Other imports
import getpass
import os
from importlib import reload
from resources import prefect_utils

# Test the DPR processing with n dummy products
output_count = 3
s3_basename = "new_zarr_product_"
s3_filenames = [f"{s3_basename}{i}" for i in range(output_count)]

# Data to test the example flow
s3_path = os.path.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "zarr"
)
my_data = {"s3_folder": s3_path, "s3_filenames": s3_filenames}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

print(f"Output zarr products will be written to: {s3_path}")

Output zarr products will be written to: s3://prefect-share/sub/dir/users/jgaucher/zarr


In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

In [5]:
# NOTE: we need to create the S3 folder with a dummy empty file before running DPR
await s3_upload_empty_file(f"{s3_path}/.empty")

12:55:19.602 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpgl38hbj4' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/zarr/.empty'.

'sub/dir/users/jgaucher/zarr/.empty'

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [28]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [29]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./dpr_processor_example.yaml"

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
12:59:01.001 | WARNING | root - Hello from 'client' '172.18.0.19' (main code)


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'dpr-flow/sprint20-dpr-example' successfully created with id      │
│ 'd079d97d-2f0d-4569-9e93-6b0882f59f36'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/d079d97d-2f0d-4569-9e93-6b0882f59f36


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'dpr-flow/sprint20-dpr-example'



In [30]:
deploy_name = "dpr-flow/sprint20-dpr-example"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'dpr-flow/sprint20-dpr-example'


## Run Prefect flow

In [31]:
print(f"Remove existing zarr products from: {s3_path!r}")
s3_delete(f"{s3_path}/{s3_basename}")

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/zarr'


[]

In [33]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'dpr-flow/sprint20-dpr-example'...
Created flow run 'quirky-sturgeon'.
└── UUID: 9a4d26da-efa2-4bb8-b2cb-cec2e7fa92ca
└── Parameters: {'s3_folder': 's3://prefect-share/sub/dir/users/jgaucher/zarr', 's3_filenames': ['new_zarr_product_0', 'new_zarr_product_1', 'new_zarr_product_2']}
└── Job Variables: {}
└── Scheduled start time: 2025-02-25 12:59:19 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/9a4d26da-efa2-4bb8-b2cb-cec2e7fa92ca
Watching flow run 'quirky-sturgeon'...


12:59:19.853 | INFO    | prefect - Flow run is in state 'Scheduled'
12:59:24.874 | INFO    | prefect - Flow run is in state 'Scheduled'
12:59:29.899 | INFO    | prefect - Flow run is in state 'Scheduled'


12:59:33.879 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:39939

12:59:33.961 | WARNING | Task run 'single_dpr_task-2ba' - Hello from 'dask' '172.18.0.4' (task 'new_zarr_product_0')

12:59:34.112 | INFO    | eopf.store.zarr - Writing prefect-share/sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/None and zarr kwargs {}

...

13:14:00.767 | CRITICAL | Task run 'single_dpr_task-da8' - hello

13:14:00.768 | WARNING | Task run 'single_dpr_task-da8' - 2025-02-25 12:36:13,016 : INFO : s1_l0_processor : run : 78 : (Process Details : (367, Dask Worker process (from Nanny)), Thread Details : (140542589535936, Dask-Default-Threads-367-0))
Log : Running S1L0Processor 
2025-02-25 12:38:28,392 : INFO : config : load_file : 260 : (Process Details : (367, Dask Worker process (from Nanny)), Thread Details : (140542589535936, Dask-Default-Threads-367-0))
Log : Registering EOConfig file : ./iw_configuration.yaml
2025-02-25 12:38:28,486 : INFO : s1_l0_processor : run : 78 : (Process Details : (367, Dask Worker process (from Nanny)), Thread Details : (140542589535936, Dask-Default-Threads-367-0))
Log : Running S1L0Processor 
2025-02-25 13:14:00,095 : INFO : s1_l0_processor : run : 78 : (Process Details : (1677, MainProcess), Thread Details : (125371178466176, MainThread))
Log : Running S1L0Processor

13:14:00.771 | INFO    | Task run 'all_my_eopf_code' - Finished in state Completed()

13:14:00.854 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:39939

13:15:50.381 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

NOTE: we could also call the Prefect flow from Python code.
This is useful to debug or see the generated HTML representation.

In [11]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import dpr_processor_example
    reload(dpr_processor_example)
    
    # Run the flow
    results = dpr_processor_example.dpr_flow(**my_data)
    
    # Display HTML representation
    import IPython
    for result in results:
        display(IPython.display.HTML(result))
    del results

## Check results

In [12]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await s3_download_directory(f"{s3_path}", local_dir)
!ls -al "$local_dir"

NameError: name 's3_download_directory' is not defined

In [ ]:
# Open them with the zarr python package
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
!pip install zarr
import zarr

for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

## 3. Shutdown the dask clusters

In [ ]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.